# 🧠 Notebook 1: Prompting Fundamentals
## Designing Effective Prompts with GPT-4o-mini

**Topics Covered:**
- Zero-shot Prompting
- Few-shot Prompting
- Chain-of-Thought (CoT) Prompting
- ReAct (Reasoning + Acting) Prompting
- Tree-of-Thought (ToT) Prompting
- Structured Prompting

**Model:** `gpt-4o-mini` | **Embeddings:** `text-embedding-3-small`

---
> 🔑 **Instructions:** Run cells top-to-bottom. Add your OpenAI API key when prompted.

In [ ]:
# ============================================
# 📦 Setup & Installation
# ============================================
!pip install openai tiktoken rich IPython -q

import os
import json
import time
from getpass import getpass
from openai import OpenAI
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table
from IPython.display import display, HTML, Markdown as IPyMarkdown

# --- API Key ---
os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API key: ")
client = OpenAI()
console = Console()

MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

def chat(messages, temperature=0.7, max_tokens=1024):
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

def embed(text):
    """Get embedding for a text string."""
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding

def show(title, content):
    """Pretty-print a response."""
    display(IPyMarkdown(f"### {title}\n---\n{content}"))

print("✅ Setup complete! Model:", MODEL, "| Embedding model:", EMBED_MODEL)

---
## 1️⃣ Zero-Shot Prompting

**Zero-shot** means giving the model a task with **no examples** — relying entirely on the model's pre-trained knowledge.

**When to use:** Simple tasks, classification, translation, summarization where the task is clear.

In [ ]:
# ============================================
# 🔹 1.1 Basic Zero-Shot
# ============================================
prompt = "Classify the following movie review as POSITIVE, NEGATIVE, or NEUTRAL.\n\nReview: 'The cinematography was breathtaking but the plot felt hollow and predictable.'\n\nClassification:"

response = chat([{"role": "user", "content": prompt}], temperature=0)
show("Zero-Shot Classification", f"**Prompt:**\n```\n{prompt}\n```\n\n**Response:** {response}")

In [ ]:
# ============================================
# 🔹 1.2 Zero-Shot with Role Assignment
# ============================================
messages = [
    {"role": "system", "content": "You are an expert linguist who analyzes text sentiment with nuance. Provide a sentiment label AND a confidence score (0-100)."},
    {"role": "user", "content": "Analyze: 'I waited 45 minutes for cold food. The waiter was apologetic though and comped our dessert.'"}
]

response = chat(messages, temperature=0)
show("Zero-Shot with System Role", f"**Response:**\n{response}")

In [ ]:
# ============================================
# 🔹 1.3 Zero-Shot Comparison — Vague vs Specific
# ============================================
vague_prompt = "Tell me about Python."
specific_prompt = "Explain 3 key advantages of Python for data science beginners, with one concrete example each. Keep it under 150 words."

r_vague = chat([{"role": "user", "content": vague_prompt}], temperature=0.5)
r_specific = chat([{"role": "user", "content": specific_prompt}], temperature=0.5)

show("❌ Vague Zero-Shot", r_vague[:500] + "...")
show("✅ Specific Zero-Shot", r_specific)

---
## 2️⃣ Few-Shot Prompting

**Few-shot** provides **examples** of input-output pairs before the actual task, teaching the model the desired format and reasoning pattern.

**When to use:** When you need consistent formatting, domain-specific behavior, or the zero-shot output isn't structured enough.

In [ ]:
# ============================================
# 🔹 2.1 Few-Shot Classification
# ============================================
few_shot_prompt = """Classify each customer support ticket into a category.

Ticket: "My order #4521 hasn't arrived after 2 weeks"
Category: Shipping

Ticket: "The zipper broke on the jacket after one use"  
Category: Product Quality

Ticket: "I was charged twice for the same item"
Category: Billing

Ticket: "Can I change the color of my order before it ships?"
Category: Order Modification

Ticket: "The app keeps crashing when I try to checkout"
Category:"""

response = chat([{"role": "user", "content": few_shot_prompt}], temperature=0)
show("Few-Shot Classification", f"**Model classified as:** {response}")

In [ ]:
# ============================================
# 🔹 2.2 Few-Shot for Structured Output (JSON)
# ============================================
json_prompt = """Extract product information into JSON format.

Input: "The Nike Air Max 90 in white/black colorway retails for $130 and comes in sizes 7-13."
Output: {"product": "Nike Air Max 90", "color": "white/black", "price": 130, "sizes": "7-13"}

Input: "Grab the Sony WH-1000XM5 noise-cancelling headphones for $349.99, available in black and silver."
Output: {"product": "Sony WH-1000XM5", "color": ["black", "silver"], "price": 349.99, "sizes": null}

Input: "The Patagonia Better Sweater in navy blue is on sale for $99, available in XS through XXL."
Output:"""

response = chat([{"role": "user", "content": json_prompt}], temperature=0)
show("Few-Shot JSON Extraction", f"```json\n{response}\n```")

In [ ]:
# ============================================
# 🔹 2.3 Shot Count Experiment (0 vs 1 vs 3 vs 5)
# ============================================
examples = [
    ("The sunset painted the sky in hues of amber.", "Nature/Poetic"),
    ("Revenue increased 23% quarter over quarter.", "Business/Finance"),
    ("The patient presented with acute respiratory distress.", "Medical"),
    ("The defendant filed a motion to dismiss.", "Legal"),
    ("The neural network achieved 97% accuracy on the test set.", "Technology/AI"),
]

test_input = "The enzyme catalyzed the hydrolysis of ATP into ADP and inorganic phosphate."

results = {}
for n_shots in [0, 1, 3, 5]:
    prompt = "Classify the following text into a domain category.\n\n"
    for ex_in, ex_out in examples[:n_shots]:
        prompt += f'Text: "{ex_in}"\nCategory: {ex_out}\n\n'
    prompt += f'Text: "{test_input}"\nCategory:'
    
    results[f"{n_shots}-shot"] = chat([{"role": "user", "content": prompt}], temperature=0)

# Display results
display(IPyMarkdown("### Shot Count Comparison"))
for k, v in results.items():
    print(f"  {k:>6s} → {v.strip()}")

---
## 3️⃣ Chain-of-Thought (CoT) Prompting

**CoT** asks the model to **reason step-by-step** before giving a final answer. This dramatically improves performance on math, logic, and multi-step reasoning tasks.

**When to use:** Math problems, logical reasoning, complex analysis, debugging.

In [ ]:
# ============================================
# 🔹 3.1 CoT vs Direct — Math Problem
# ============================================
math_problem = """A store sells notebooks for $3.50 each. If you buy 5 or more, you get a 15% discount.
Sales tax is 8.5%. How much do you pay for 7 notebooks?"""

# Direct approach
direct_response = chat([
    {"role": "user", "content": f"{math_problem}\n\nGive the final answer only."}
], temperature=0)

# CoT approach
cot_response = chat([
    {"role": "user", "content": f"{math_problem}\n\nThink step by step, showing all calculations before giving the final answer."}
], temperature=0)

show("❌ Direct Answer", direct_response)
show("✅ Chain-of-Thought", cot_response)

In [ ]:
# ============================================
# 🔹 3.2 CoT for Logical Reasoning
# ============================================
logic_problem = """At a dinner party:
- Alice sits next to Bob
- Carol sits across from Alice
- Dave sits next to Carol but not next to Bob
- Eve sits at the remaining seat

The table is rectangular with 3 seats on each long side.
Who sits next to Eve?

Think through this step by step, drawing out the seating arrangement."""

response = chat([{"role": "user", "content": logic_problem}], temperature=0)
show("CoT — Logical Reasoning", response)

In [ ]:
# ============================================
# 🔹 3.3 Auto-CoT with "Let's think step by step"
# ============================================
problems = [
    "If a shirt costs $25 and is on sale for 40% off, and then you have a $5 coupon, what do you pay?",
    "A train leaves Station A at 9:00 AM going 60 mph. Another leaves Station B (300 miles away) at 10:00 AM going 90 mph toward Station A. When do they meet?",
    "In a class of 30 students, 18 play soccer, 15 play basketball, and 10 play both. How many play neither?"
]

for i, problem in enumerate(problems, 1):
    response = chat([
        {"role": "system", "content": "You are a math tutor. Always think step by step. Show your work clearly."},
        {"role": "user", "content": problem}
    ], temperature=0)
    show(f"CoT Problem {i}", response)
    print()

---
## 4️⃣ ReAct Prompting (Reasoning + Acting)

**ReAct** combines **reasoning** (thinking about what to do) with **acting** (taking actions like searching, calculating). The model alternates between Thought → Action → Observation cycles.

**When to use:** Complex tasks needing external tools, multi-step research, agentic workflows.

In [ ]:
# ============================================
# 🔹 4.1 Simulated ReAct Pattern
# ============================================
react_prompt = """You are an AI assistant that solves problems using the ReAct framework.
For each step, output:
  Thought: [your reasoning about what to do next]
  Action: [the action you take — Search, Calculate, or Lookup]
  Observation: [what you learned from the action]

Repeat until you can give a Final Answer.

Question: What is the population density of the country that won the 2022 FIFA World Cup, 
and how does it compare to the world average?"""

response = chat([{"role": "user", "content": react_prompt}], temperature=0.3)
show("ReAct Pattern", response)

In [ ]:
# ============================================
# 🔹 4.2 ReAct with Actual Tool Simulation
# ============================================
import re

tools = {
    "calculate": lambda expr: str(eval(expr)),
    "lookup_population": lambda country: {"Argentina": {"pop": 46_044_703, "area_km2": 2_780_400}}.get(country, {"pop": "unknown", "area_km2": "unknown"}),
}

messages = [
    {"role": "system", "content": """You are a ReAct agent. You have access to these tools:
- calculate(expression): Evaluate a math expression
- lookup_population(country): Get population and area data

For each step, output EXACTLY this format:
Thought: <reasoning>
Action: <tool_name>(<argument>)

Wait for the Observation before continuing. When done, output:
Final Answer: <your answer>"""},
    {"role": "user", "content": "What is the population density of Argentina in people per square km?"}
]

# Run 3 reasoning cycles
for cycle in range(3):
    response = chat(messages, temperature=0)
    messages.append({"role": "assistant", "content": response})
    
    # Extract action
    action_match = re.search(r'Action:\s*(\w+)\((.+?)\)', response)
    if action_match:
        tool_name, arg = action_match.group(1), action_match.group(2).strip('"\' ')
        if tool_name in tools:
            result = tools[tool_name](arg)
            observation = f"Observation: {result}"
            messages.append({"role": "user", "content": observation})
            print(f"Cycle {cycle+1}: {tool_name}({arg}) → {result}")
    
    if "Final Answer:" in response:
        break

show("ReAct Agent Result", response)

---
## 5️⃣ Tree-of-Thought (ToT) Prompting

**ToT** explores **multiple reasoning paths** simultaneously, evaluates each, and selects the best. Think of it as CoT with branching and backtracking.

**When to use:** Creative problems, strategic planning, puzzles, decisions with trade-offs.

In [ ]:
# ============================================
# 🔹 5.1 Tree-of-Thought — Strategy Problem
# ============================================
tot_prompt = """You are solving a complex problem using Tree-of-Thought reasoning.

Problem: A startup has $500K in funding and needs to decide how to allocate it across:
hiring (engineers cost $120K/yr, marketers $90K/yr), product development tools ($50K), 
and marketing campaigns ($30K-$100K). They need to launch an MVP in 6 months.

Generate 3 DIFFERENT strategic approaches (branches):

**Branch A:** [Engineering-heavy approach]
**Branch B:** [Balanced approach]  
**Branch C:** [Marketing-heavy approach]

For EACH branch:
1. Describe the allocation
2. List 2 pros and 2 cons
3. Rate the chance of successful MVP launch (0-100%)

Then EVALUATE all branches and select the best one with justification."""

response = chat([{"role": "user", "content": tot_prompt}], temperature=0.5, max_tokens=2000)
show("Tree-of-Thought — Strategic Planning", response)

In [ ]:
# ============================================
# 🔹 5.2 ToT — Creative Problem Solving
# ============================================
tot_creative = """Use Tree-of-Thought reasoning to solve this puzzle:

You have a 3x3 grid. Place the numbers 1-9 so that:
- Each row sums to 15
- Each column sums to 15
- Each diagonal sums to 15

Explore at least 2 different starting approaches:
- For each approach, show your reasoning step by step
- If an approach hits a dead end, backtrack and try another path
- Mark which approach succeeds"""

response = chat([{"role": "user", "content": tot_creative}], temperature=0.3, max_tokens=2000)
show("ToT — Magic Square Puzzle", response)

---
## 6️⃣ Structured Prompting

**Structured prompting** uses **explicit formatting** (XML tags, markdown templates, tables, JSON schemas) to constrain and organize both input and output.

**When to use:** API responses, data extraction, consistent report generation, any task needing reliable formatting.

In [ ]:
# ============================================
# 🔹 6.1 XML-Tagged Structured Prompt
# ============================================
structured_prompt = """Analyze the following product review using the exact XML structure below.

<review>
I bought this wireless keyboard for my home office. The typing feel is surprisingly good 
for the price point ($45), and battery life has been excellent — going on 3 months without 
a change. However, the Bluetooth connection drops occasionally, maybe once a day, which 
is frustrating during meetings. Build quality feels a bit cheap, plastic-y. Overall decent 
value but not premium.
</review>

Respond ONLY with this XML structure:
<analysis>
  <sentiment score="-1.0 to 1.0">LABEL</sentiment>
  <key_pros>
    <pro>...</pro>
  </key_pros>
  <key_cons>
    <con>...</con>
  </key_cons>
  <topics>
    <topic relevance="high|medium|low">...</topic>
  </topics>
  <summary max_words="25">...</summary>
  <recommendation>BUY | CONSIDER | SKIP</recommendation>
</analysis>"""

response = chat([{"role": "user", "content": structured_prompt}], temperature=0)
show("Structured XML Output", f"```xml\n{response}\n```")

In [ ]:
# ============================================
# 🔹 6.2 Markdown Template Prompting
# ============================================
template_prompt = """Generate a competitive analysis using this EXACT template. Fill every section.

# Competitive Analysis: [Product Category]

## Executive Summary
> [2-3 sentence overview]

## Comparison Matrix

| Feature | Company A | Company B | Company C |
|---------|-----------|-----------|-----------|
| Price | | | |
| Key Strength | | | |
| Key Weakness | | | |
| Target Market | | | |
| Rating (1-5) | | | |

## Strategic Recommendations
1. **Short-term (0-3 months):** [action]
2. **Mid-term (3-12 months):** [action]  
3. **Long-term (1-3 years):** [action]

## Risk Assessment
- 🔴 High Risk: [describe]
- 🟡 Medium Risk: [describe]
- 🟢 Low Risk: [describe]

---
Topic: Cloud-based project management tools (Asana vs Monday.com vs ClickUp)"""

response = chat([{"role": "user", "content": template_prompt}], temperature=0.3, max_tokens=2000)
show("Structured Markdown Template", response)

In [ ]:
# ============================================
# 🔹 6.3 Using Embeddings for Semantic Similarity
# ============================================
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare how different prompting techniques describe the same concept
descriptions = {
    "Zero-shot": "Classify this text as positive or negative.",
    "Few-shot": "Here are examples: 'Great!' → positive, 'Terrible!' → negative. Now classify: 'Amazing product!'",
    "CoT": "Let me think step by step about the sentiment. The word 'amazing' is very positive, so this is positive.",
    "Structured": "<task>sentiment_analysis</task><input>Amazing product!</input><output format='json'>{"label": "positive", "confidence": 0.95}</output>"
}

embeddings = {k: embed(v) for k, v in descriptions.items()}

print("📐 Semantic Similarity Between Prompting Techniques\n")
techniques = list(embeddings.keys())
for i in range(len(techniques)):
    for j in range(i+1, len(techniques)):
        sim = cosine_similarity(embeddings[techniques[i]], embeddings[techniques[j]])
        bar = "█" * int(sim * 30)
        print(f"  {techniques[i]:>12s} ↔ {techniques[j]:<12s}  {sim:.4f}  {bar}")

---
## 🎯 Summary & Key Takeaways

| Technique | Best For | Complexity | Reliability |
|-----------|----------|------------|-------------|
| **Zero-shot** | Simple, well-defined tasks | ⭐ | ⭐⭐⭐ |
| **Few-shot** | Format consistency, domain tasks | ⭐⭐ | ⭐⭐⭐⭐ |
| **CoT** | Math, logic, multi-step reasoning | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **ReAct** | Agentic tasks, tool use | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **ToT** | Creative problems, trade-off analysis | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Structured** | APIs, data extraction, reports | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |

> 💡 **Next:** Move to **Notebook 2** for hands-on prompt building!